### References

*   **GEKKO Optimization Suite Article:**
    Beal, L. D. R., Hill, D., Martin, R. A., & Hedengren, J. D. (2018). *GEKKO Optimization Suite*. Processes, 6(8), 106. doi.org

*   **APMonitor Course Website (ME575):**
    Hedengren, J. D. (n.d.). *Design Optimization (ME 575)*. APMonitor. Retrieved January 9, 2026, from https://apmonitor.com/me575/index.php/Main/HomePage

*   **Mr. P Solver (YouTube Channel):**
    Mr. P Solver. (n.d.). *Home* [YouTube channel]. YouTube. Retrieved January 9, 2026, from https://www.youtube.com/@MrPSolver

*   **The OR Book (Textbook):**
    Winston, W. L. (2004). *Operations Research: Applications and Algorithms* (4th ed.). Thomson Brooks/Cole.


In [1]:
import numpy as np

# solving diet as blending problem 

In [4]:
# solving diet problem as blending problem but diet problem has GEPri@x >= GEPi not GEPri@x >= GEPi*zs

#products
SP=np.array([0]) #parhapd if you want to sell your food
D=np.array([0])
Ad=np.array([0])

GEPi=np.array([[500],
               [6],
               [10],
               [8]
              ])
#BEPi=np.array([[0.01,0.02,0.01]])

#Resources
BP=np.array([50,20,30,80])
MP=np.array([5000,5000,5000,5000]) # set this to high number if there is infinity of resourses 
GEPRi=np.array([[400,200,150,500],
                [3,2,0,0],
                [2,2,4,4],
                [2,4,1,5]
               ])
#BEPRi=np.array([[0.005,0.02,0.03]])

RPC=0
maxP=14000

p=len(SP)
r=len(BP)

from gekko import GEKKO
m = GEKKO(remote=False)

x = m.Array(m.Var,(r,p),lb=0)
a = m.Array(m.Var,p,lb=0)
xs=[]
cost=[]


for i in range(r):
    xs.append(m.Intermediate(m.sum(x[i])))
    cost.append(m.Intermediate(BP[i]*xs[i] ))
    m.Equations( [ xs[i]<=MP[i]  ])
m.Equations( [ m.sum(xs)<= maxP  ])


#revenue=[]
#z=x.T
z=[]
zs=[]


for j in range(p):
    zc=[]
    for i in range(r):
        zc.append(x[i][j])
    z.append(zc)
    zs.append(m.Intermediate(m.sum(zc) ))
#    revenue.append(m.Intermediate( SP[j]*zs[j]))
#    m.Equations( [ zs[j]==D[j]+ Ad[j]*a[j] ])

    for i in range(len(GEPi)):# Good elements
        #since matric multiplication is just element multiplication then addition
#        m.Equations( [ m.sum( GEPRi[i]*z[j])>=GEPi[i][j]*zs[j]   ])  
        m.Equations( [ m.sum( GEPRi[i]*z[j])>=GEPi[i][j]   ]) #here relationship is different for zero selling price perhaps
        
    #bad elements
#    for i in range(len(BEPi)):
#        m.Equations( [ m.sum( BEPRi[i]*z[j])<=BEPi[i][j]*zs[j]   ])

#Ad_cost=m.Intermediate(m.sum(a))
costT=m.Intermediate(m.sum(cost))
#tatal_revenue=m.Intermediate(m.sum(revenue))
#production=m.Intermediate(m.sum(xs)*RPC)
#profit=m.Intermediate(tatal_revenue-Ad_cost-production-costT)

m.Minimize(costT)
m.options.SOLVER=1
m.solve(disp=False)
print('Cost= ' + str(costT.value[0]))
x

Cost= 90.0


array([[[0.0]],
       [[3.0]],
       [[1.0]],
       [[0.0]]], dtype=object)

In [7]:
p=1
for j in range(p):
    print(j)

0


In [9]:
# solving diet problem as blending problem but diet problem has GEPri@x >= GEPi not GEPri@x >= GEPi*zs...a more detailed appraoch

#products
SP=np.array([0]) #parhaps if you want to sell your food ,and note: some problems might be infeasable
D=np.array([0])
Ad=np.array([0])

GEPi=np.array([[500],
               [6],
               [10],
               [8]
              ])
BEPi=np.array([]) #no bad element for now

#Resources
BP=np.array([50,20,30,80])
MP=np.array([5000,5000,5000,5000]) # set this to high number if there is infinity of resourses 
GEPRi=np.array([[400,200,150,500],
                [3,2,0,0],
                [2,2,4,4],
                [2,4,1,5]
               ])
BEPRi=np.array([]) #no bad elements in any resource

RPC=0
maxP=14000

p=len(SP)
r=len(BP)

from gekko import GEKKO
m = GEKKO(remote=False)

x = m.Array(m.Var,(r,p),lb=0)
a = m.Array(m.Var,p,lb=0)
xs=[]
cost=[]


for i in range(r):
    xs.append(m.Intermediate(m.sum(x[i])))
    cost.append(m.Intermediate(BP[i]*xs[i] ))
    m.Equations( [ xs[i]<=MP[i]  ])
m.Equations( [ m.sum(xs)<= maxP  ])

#if you dont decleare these, problems might arise
revenue=[] 
#z=x.T
z=[]
zs=[]


for j in range(p):
    zc=[]
    for i in range(r):
        zc.append(x[i][j])
    z.append(zc)
    zs.append(m.Intermediate(m.sum(zc) )) #zs might be the amount of food
    revenue.append(m.Intermediate( SP[j]*zs[j])) # so here might be another difference in blending with diet
#    revenue.append(m.Intermediate( SP[j]) ) # so here is another difference in blending with diet?
#    m.Equations( [ zs[j]==D[j]+ Ad[j]*a[j] ])

    for i in range(len(GEPi)):# Good elements
        #since matric multiplication is just element multiplication then addition
#        m.Equations( [ m.sum( GEPRi[i]*z[j])>=GEPi[i][j]*zs[j]   ]) 
        m.Equations( [ m.sum( GEPRi[i]*z[j])>=GEPi[i][j]   ]) #here relationship is different, but we can concider the amount of food zs
        
    #bad elements
    for i in range(len(BEPi)):
#        m.Equations( [ m.sum( BEPRi[i]*z[j])<=BEPi[i][j]*zs[j]   ])
        m.Equations( [ m.sum( BEPRi[i]*z[j])<=BEPi[i][j]   ])

Ad_cost=m.Intermediate(m.sum(a))
costT=m.Intermediate(m.sum(cost))
tatal_revenue=m.Intermediate(m.sum(revenue)) 
production=m.Intermediate(m.sum(xs)*RPC)
profit=m.Intermediate(tatal_revenue-Ad_cost-production-costT)


m.Maximize(profit)
m.options.SOLVER=1
m.solve(disp=False)
print('Cost= ' + str(-profit.value[0]))
print('x  is:')
print(x)
print("ad a is:")
print(a)
print (" your 'amount of food' is")
zs

Cost= 90.0
x  is:
[[[0.0]]
 [[3.0]]
 [[1.0]]
 [[0.0]]]
ad a is:
[[0.0]]
 your 'amount of food' is


[[4.0]]

In [13]:
zs

[[4.0]]

In [17]:
#a play code
# solving diet problem as blending problem but diet problem has GEPri@x >= GEPi not GEPri@x >= GEPi*zs...a more detailed appraoch

#products
SP=np.array([0]) #parhaps if you want to sell your food, and note: some problems might be infeasable
D=np.array([4])
Ad=np.array([0])

GEPi=np.array([[500],
               [6],
               [10],
               [8]
              ])
BEPi=np.array([]) #no bad element for now

#Resources
BP=np.array([50,20,30,80])
MP=np.array([5000,5000,5000,5000]) # set this to high number if there is infinity of resourses 
GEPRi=np.array([[400,200,150,500],
                [3,2,0,0],
                [2,2,4,4],
                [2,4,1,5]
               ])
BEPRi=np.array([]) #no bad elements in any resource

RPC=0
maxP=14000

p=len(SP)
r=len(BP)

from gekko import GEKKO
m = GEKKO(remote=False)

x = m.Array(m.Var,(r,p),lb=0)
a = m.Array(m.Var,p,lb=0)
xs=[]
cost=[]


for i in range(r):
    xs.append(m.Intermediate(m.sum(x[i])))
    cost.append(m.Intermediate(BP[i]*xs[i] ))
    m.Equations( [ xs[i]<=MP[i]  ])
m.Equations( [ m.sum(xs)<= maxP  ])

#if you dont decleare these, problems might arise
revenue=[] 
#z=x.T
z=[]
zs=[]


for j in range(p):
    zc=[]
    for i in range(r):
        zc.append(x[i][j])
    z.append(zc)
    zs.append(m.Intermediate(m.sum(zc) )) #zs might be the amount of food
    revenue.append(m.Intermediate( SP[j]*zs[j])) # so here might be another difference in blending with diet
#    revenue.append(m.Intermediate( SP[j]) ) # so here is another difference in blending with diet?
#    m.Equations( [ zs[j]==D[j]+ Ad[j]*a[j] ])

    for i in range(len(GEPi)):# Good elements
        #since matric multiplication is just element multiplication then addition
#        m.Equations( [ m.sum( GEPRi[i]*z[j])>=GEPi[i][j]*zs[j]   ]) 
        m.Equations( [ m.sum( GEPRi[i]*z[j])>=GEPi[i][j]   ]) #here relationship is different, but we can concider the amount of food zs
        
    #bad elements
    for i in range(len(BEPi)):
#        m.Equations( [ m.sum( BEPRi[i]*z[j])<=BEPi[i][j]*zs[j]   ])
        m.Equations( [ m.sum( BEPRi[i]*z[j])<=BEPi[i][j]   ])

Ad_cost=m.Intermediate(m.sum(a))
costT=m.Intermediate(m.sum(cost))
tatal_revenue=m.Intermediate(m.sum(revenue)) 
production=m.Intermediate(m.sum(xs)*RPC)
profit=m.Intermediate(tatal_revenue-Ad_cost-production-costT)


m.Maximize(profit)
m.options.SOLVER=1
m.solve(disp=False)
print('Cost= ' + str(-profit.value[0]))
print('x  is:')
print(x)
print("ad a is:")
print(a)
print (" your 'amount of food' is")
zs

Cost= 90.0
x  is:
[[[0.0]]
 [[3.0]]
 [[1.0]]
 [[0.0]]]
ad a is:
[[0.0]]
 your 'amount of food' is


[[4.0]]

In [19]:
len([[]])

1

In [21]:
for i in range(len([])):
    print(i)

# now generating a matrix similar to table14 pg107 in OR book

other usefull pages : pg 314

In [33]:
import numpy as np
#blending problem again as : max Cx st Ax >= b

#products
SP=np.array([70,60,50])
D=np.array([3000,2000,1000])
Ad=np.array([10,10,10])
GEPi=np.array([[10,8,6]])
BEPi=np.array([[0.01,0.02,0.01]])

#Resources
BP=np.array([45,35,25])
MP=np.array([5000,5000,5000])
GEPRi=np.array([[12,6,8]])
BEPRi=np.array([[0.005,0.02,0.03]])

RPC=4
maxP=14000

p=len(SP)
r=len(BP)




C=[0]*(r*p)
for j in range(p):
    for i in range(r):
        Id=j + p*(i)
        #C[ (j + p*(i)) ]+=SP[j]-BP[i]-RPC*(1)
        C[ (j + p*(i)) ]=SP[j]-BP[i]-RPC*(1) #C already creted without the ads


A=[]
Eq=[0]*len(C)
index=0
b=[]
a_index_g=[] # for A[i]x >= b[i]
a_index_l=[] # for -A[i]x >= -b[i]

for j in range(p): #z_s[i] = D[i]
    A.append(Eq.copy()) ##In Python, when you create a list of lists using certain methods (like matrix = [row] * N or appending the same list object in a loop), you are filling the outer list with multiple references to the exact same inner list in memory. 
    A.append(Eq.copy())
    b.append(D[j] )
    b.append(-D[j] )
    for i in range(r):
        A[ index ][ (j + p*(i))]=1
        A[ (index +1 ) ][ (j + p*(i))]=-1
    a_index_g.append(index) #here is where we have z_s[i] = D[i] + 10xa
    a_index_l.append(index+1) #for the additional equqtion
   
    index=index+2

for i in range(r): #x_s[i] <= MP[i]
    A.append(Eq.copy())
    b.append( -MP[i] )
    for j in range(p):
        A[ index ][ (j + p*(i))]=-1
    
    index+=1

#good products
for n in range(len(GEPi)):
    for j in range(p):
        A.append(Eq.copy())
        b.append(0)
        for i in range(r):
            A[ index ][ (j + p*(i))]= GEPRi[n][i]-GEPi[n][j]
        index+=1

#bad products
for n in range(len(BEPi)):
    for j in range(p):
        A.append(Eq.copy())
        A.append(Eq.copy())
        b.append(0)
        b.append(0)
        for i in range(r):
            A[ index ][ (j + p*(i))]= BEPRi[n][i]-BEPi[n][j]
            A[ index+1 ][ (j + p*(i))]= -BEPRi[n][i]+BEPi[n][j]
        index+=2

#now we deal with ads
for i in range(len(Ad)): # first append C
    C.append(-1)
for i in range(len(A)):
    for j in range(len(Ad)): #adding zero to each elemeent of A
        A[i].append(0) 

diff=len(C)-len(Ad)
rdiff=len(C)-p*r
for i in range( len(a_index_g)):# this was the index where we had equality constaint so we shall also modify the equation following the index too
    A[(a_index_g[i])][(diff+i) ]=-Ad[i] #Ad is also leng
    #print( A[(a_index_g[i])])
for i in range( len(a_index_l) ):
    A[(a_index_l[i])][(diff+i) ]=Ad[i]
    #print( A[(a_index_l[i])]  )       
# by now we have A,C,b

#here we used Max Cx, st Ax >= b

m = GEKKO(remote=False)
x = m.Array(m.Var,len(C),lb=0)
m.Maximize(C@x)
#m.Minimize(C@x)
Ax = A@x
m.Equations([Ax[i]>=b[i] for i in range(len(b))])
m.options.SOLVER=1 #to make sure its integer
m.solve(disp=False)

# Print the results
print("Objective function: ", -m.options.objfcnval)
print("Optimal solution: ", x)


Objective function:  287750.0
Optimal solution:  [[2028.2691395] [2305.0641939] [666.66666667] [929.32715131]
 [3737.3395154] [333.33333333] [42.403709215] [3457.5962908] [0.0] [0.0]
 [750.0] [0.0]]


In [36]:
a_index_g,a_index_l

([0, 2, 4], [1, 3, 5])

In [39]:
len(A),len(b)

(18, 18)

In [41]:

len(A[17]),len(C)

(12, 12)

In [43]:
b

[np.int64(3000),
 np.int64(-3000),
 np.int64(2000),
 np.int64(-2000),
 np.int64(1000),
 np.int64(-1000),
 np.int64(-5000),
 np.int64(-5000),
 np.int64(-5000),
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [45]:
A


[[1, 0, 0, 1, 0, 0, 1, 0, 0, np.int64(-10), 0, 0],
 [-1, 0, 0, -1, 0, 0, -1, 0, 0, np.int64(10), 0, 0],
 [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, np.int64(-10), 0],
 [0, -1, 0, 0, -1, 0, 0, -1, 0, 0, np.int64(10), 0],
 [0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, np.int64(-10)],
 [0, 0, -1, 0, 0, -1, 0, 0, -1, 0, 0, np.int64(10)],
 [-1, -1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, -1, -1, -1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, -1, -1, -1, 0, 0, 0],
 [np.int64(2), 0, 0, np.int64(-4), 0, 0, np.int64(-2), 0, 0, 0, 0, 0],
 [0, np.int64(4), 0, 0, np.int64(-2), 0, 0, np.int64(0), 0, 0, 0, 0],
 [0, 0, np.int64(6), 0, 0, np.int64(0), 0, 0, np.int64(2), 0, 0, 0],
 [np.float64(-0.005),
  0,
  0,
  np.float64(0.01),
  0,
  0,
  np.float64(0.019999999999999997),
  0,
  0,
  0,
  0,
  0],
 [np.float64(0.005),
  0,
  0,
  np.float64(-0.01),
  0,
  0,
  np.float64(-0.019999999999999997),
  0,
  0,
  0,
  0,
  0],
 [0,
  np.float64(-0.015),
  0,
  0,
  np.float64(0.0),
  0,
  0,
  np.float64(0.009999999999999998),
 

In [47]:
C

[np.int64(21),
 np.int64(11),
 np.int64(1),
 np.int64(31),
 np.int64(21),
 np.int64(11),
 np.int64(41),
 np.int64(31),
 np.int64(21),
 -1,
 -1,
 -1]

# cleanup

In [53]:
m.cleanup()